# Schedule Market Calendar Gate v1

目标：把交易日、半日市、休市判断沉到 `src/schedule`，让数据更新、模型刷新、dashboard export 和 paper/sim 执行在进入任务前先有稳定 gate。

In [1]:
from pathlib import Path
import json
import sys
import tempfile

import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ScheduleJob, classify_market_calendar_day, run_job_once

OUTPUT_DIR = REPO_ROOT / "research" / "schedule_market_calendar_gate_v1"
trading_calendar = ("2026-01-02", "2026-01-05", "2026-01-06")
half_days = ("2026-01-02",)
holiday_reasons = {"2026-01-01": "New Year's Day"}

decisions = [
    classify_market_calendar_day("2026-01-01", trading_calendar=trading_calendar, half_days=half_days, holiday_reasons=holiday_reasons),
    classify_market_calendar_day("2026-01-02", trading_calendar=trading_calendar, half_days=half_days, holiday_reasons=holiday_reasons),
    classify_market_calendar_day("2026-01-05", trading_calendar=trading_calendar, half_days=half_days, holiday_reasons=holiday_reasons),
]

events: list[str] = []
calls = {"count": 0}
with tempfile.TemporaryDirectory() as raw_tmp:
    job = ScheduleJob(name="paper_execution", task="paper_sim", cron="manual")
    closed_result = run_job_once(
        job,
        task=lambda: calls.__setitem__("count", calls["count"] + 1) or {"should_not_run": True},
        lock_dir=Path(raw_tmp),
        notify=events.append,
        market_calendar_decision=decisions[0],
    )
    closed_lock_exists = (Path(raw_tmp) / "paper_execution.lock").exists()

decisions_frame = pd.DataFrame(decisions)
decisions_frame.to_csv(OUTPUT_DIR / "market_calendar_decisions.csv", index=False, lineterminator="\n")

summary = {
    "schema_version": "research.schedule_market_calendar_gate.v1",
    "status": "completed",
    "decision_schema_version": decisions[0]["schema_version"],
    "decision_count": int(len(decisions)),
    "market_statuses": sorted(decisions_frame["market_status"].unique().tolist()),
    "closed_status": closed_result.status,
    "closed_skip_reason": closed_result.detail,
    "closed_task_calls": int(calls["count"]),
    "closed_lock_created": bool(closed_lock_exists),
    "notification_events": events,
    "next_trading_date_from_closed": decisions[0]["next_trading_date"],
    "artifacts": {
        "market_calendar_decisions": "market_calendar_decisions.csv",
        "review": "schedule_market_calendar_gate_review.md",
    },
}
(OUTPUT_DIR / "schedule_market_calendar_gate_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(OUTPUT_DIR / "schedule_market_calendar_gate_review.md").write_text(
    "# Schedule Market Calendar Gate Review\n\n"
    "## 中文结论\n\n"
    "- 做了什么：验证 `src/schedule` 可在 job 执行前识别 normal、half_day 和 closed 市场日历状态。\n"
    "- 核心指标：closed job 返回 skipped，task 调用次数为 0，lock_created=False，且不发送成功/失败通知。\n"
    "- regime 差异：本卡片不评估市场 regime；它处理的是 schedule 日历状态差异，供后续数据更新、模型刷新、dashboard export 和 paper/sim 复用。\n"
    "- 是否晋升：否；这是 schedule 框架 gate，不是模型或策略晋升。\n"
    "- 风险：若休市或半日市没有在任务入口明确处理，定时任务可能在错误日期更新数据或触发 paper/sim。\n"
    "- 下一步：把该 gate 接入更高层 schedule pipeline 的运行前检查，并在生产 notebook 中固定真实交易日历来源。\n",
    encoding="utf-8",
)
summary


{'schema_version': 'research.schedule_market_calendar_gate.v1',
 'status': 'completed',
 'decision_schema_version': 'schedule.market_calendar_decision.v1',
 'decision_count': 3,
 'market_statuses': ['closed', 'half_day', 'normal'],
 'closed_status': 'skipped',
 'closed_skip_reason': "New Year's Day",
 'closed_task_calls': 0,
 'closed_lock_created': False,
 'notification_events': [],
 'next_trading_date_from_closed': '2026-01-02',
 'artifacts': {'market_calendar_decisions': 'market_calendar_decisions.csv',
  'review': 'schedule_market_calendar_gate_review.md'}}

## 中文结论

- 做了什么：新增并验证 `src/schedule` 的市场日历 gate，覆盖 normal、half_day 和 closed 三类调度日期。
- 核心指标：closed 日期返回 `skipped`，skip reason 记录休市原因，task 调用次数为 0，不创建 lock，不发送成功/失败通知；half_day 与 normal 日期保留可运行决策。
- regime 差异：本 notebook 不分析交易 regime；这里的差异是日历状态差异，用于避免 schedule 在休市或半日市上误触发数据、模型、dashboard 或 paper/sim 流程。
- 是否晋升：否；这是 `src/schedule` 基础契约补强，不是策略晋升。
- 风险：当前示例使用小型静态交易日历；production notebook 需要固定真实交易日历来源、半日市列表和休市原因来源。
- 下一步：把 market calendar gate 接入更高层 pipeline preflight，并让 schedule 通知摘要带上 market_status 和 next_trading_date。